<a href="https://colab.research.google.com/github/omotola2029-glitch/SentACast-/blob/main/Copy_of_Model_split_and_preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import joblib
from google.colab import files

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [ ]:
# Load the cleaned diabetes dataset

url = "https://raw.githubusercontent.com/Group-22-AI-ML/medical-diagnosis-network/main/data/cleaned_diabetes_data.csv"

df = pd.read_csv(url)

# Display the first 5 rows
df.head()

,age,gender,polyuria,polydipsia,sudden_weight_loss,weakness,polyphagia,genital_thrush,visual_blurring,obesity,class
0,40,1,0,1,0,1,0,0,0,1,1
1,58,1,0,0,0,1,0,0,1,0,1
2,41,1,1,0,0,1,1,0,0,0,1
3,45,1,0,0,1,1,1,1,0,0,1
4,60,1,1,1,1,1,1,0,1,1,1


In [ ]:
# Check number of rows and columns
print("Dataset shape:", df.shape)

# Display column names
print("\nColumns:")
print(df.columns.tolist())

# Check data types and missing values
print("\nDataset information:")
df.info()

print("Total records:", len(df))
print("Repeated records:", df.duplicated().sum())
print("Unique records:", df.drop_duplicates().shape[0])

# How many times does each complete record appear?
counts = df.value_counts()

print("Records appearing once:", (counts == 1).sum())
print("Records appearing twice:", (counts == 2).sum())
print("Records appearing 3 times:", (counts == 3).sum())
print("Records appearing 4+ times:", (counts >= 4).sum())

In [ ]:
# Check for missing values

print(df.isnull().sum())

In [ ]:
print("Number of duplicate rows:", df.duplicated().sum())

In [ ]:
# Features (X)
X = df.drop("class", axis=1)

# Target (y)
y = df["class"]

print("Features (X):")
print(X.head())

print("\nTarget (y):")
print(y.head())

In [ ]:
# Columns used to identify identical records
feature_columns = [
    "age",
    "gender",
    "polyuria",
    "polydipsia",
    "sudden_weight_loss",
    "weakness",
    "polyphagia",
    "genital_thrush",
    "visual_blurring",
    "obesity",
    "class"
]

# Create a group ID for identical records
df["group_id"] = df.groupby(feature_columns).ngroup()

# Count how many records are in each group
group_sizes = df["group_id"].value_counts()

print("Total groups:", len(group_sizes))
print("Largest group:", group_sizes.max())

print("\nGroup size distribution:")
print(group_sizes.value_counts().sort_index())

In [ ]:
print("Target class distribution:")
print(y.value_counts())

print("\nTarget class percentages:")
print(y.value_counts(normalize=True) * 100)

# Create groups of identical records
df["group_id"] = df.groupby(feature_columns).ngroup()

# Show how many records belong to each group
group_sizes = df["group_id"].value_counts()

print("Total groups:", len(group_sizes))
print("Largest group:", group_sizes.max())

In [ ]:
# Check for repeated record
is_repeated = df.duplicated(subset=feature_columns, keep=False)

repeated = df[is_repeated].copy()
unique = df[~is_repeated].copy()

print("\nRepeated records by class:")
print(repeated["class"].value_counts())

print("\nUnique records by class:")
print(unique["class"].value_counts())

In [ ]:
# Separate repeated and unique records
repeated = df[df.duplicated(keep=False)]
unique = df[~df.duplicated(keep=False)]

# Shuffle them
repeated = repeated.sample(frac=1, random_state=42)
unique = unique.sample(frac=1, random_state=42)

# Create the splits
train = pd.concat([repeated.iloc[:274], unique.iloc[:90]])
val = pd.concat([repeated.iloc[274:342], unique.iloc[90:100]])
test = pd.concat([repeated.iloc[342:], unique.iloc[100:]])

# Remove the temporary group ID
train = train.drop(columns=["group_id"])
val = val.drop(columns=["group_id"])
test = test.drop(columns=["group_id"])

# Shuffle each split
train = train.sample(frac=1, random_state=42).reset_index(drop=True)
val = val.sample(frac=1, random_state=42).reset_index(drop=True)
test = test.sample(frac=1, random_state=42).reset_index(drop=True)

print("Train:", train.shape)
print("Validation:", val.shape)
print("Test:", test.shape)

print("\nTrain class distribution:")
print(train["class"].value_counts())

print("\nValidation class distribution:")
print(val["class"].value_counts())

print("\nTest class distribution:")
print(test["class"].value_counts())

In [ ]:
# Separating X and Y

X_train = train.drop("class", axis=1)
y_train = train["class"]

X_val = val.drop("class", axis=1)
y_val = val["class"]

X_test = test.drop("class", axis=1)
y_test = test["class"]

In [ ]:
# Scale age only
scaler = StandardScaler()

X_train_scaled = X_train.copy()
X_val_scaled = X_val.copy()
X_test_scaled = X_test.copy()

X_train_scaled[['age']] = scaler.fit_transform(X_train[['age']])
X_val_scaled[['age']] = scaler.transform(X_val[['age']])
X_test_scaled[['age']] = scaler.transform(X_test[['age']])

print("\nScaled data:")
print("X_train:", X_train_scaled.shape)
print("X_val:", X_val_scaled.shape)
print("X_test:", X_test_scaled.shape)

In [ ]:
# Save scaled data splits
train_final = X_train_scaled.copy()
train_final["class"] = y_train.values

val_final = X_val_scaled.copy()
val_final["class"] = y_val.values

test_final = X_test_scaled.copy()
test_final["class"] = y_test.values

train_final.to_csv("train.csv", index=False)
val_final.to_csv("validation.csv", index=False)
test_final.to_csv("test.csv", index=False)

print("All data splits saved successfully.")

# Auto-download the data splits
files.download("train.csv")
files.download("validation.csv")
files.download("test.csv")


# Save scaler
joblib.dump(scaler, "diabetes_age_scaler.pkl")
print("Scaler saved as diabetes_age_scaler.pkl")

# Save feature order
joblib.dump(X_train.columns.tolist(), "diabetes_features.pkl")

# Auto-download the PKL files
files.download("diabetes_age_scaler.pkl")
files.download("diabetes_features.pkl")

print("diabetes_age_scaler.pkl downloaded to device")
print("diabetes_features.pkl downloaded to device")